In [0]:
# Databricks notebook source
# CELL 1: Discovery Configuration, Exclusions & Setup
from sodapy import Socrata
from pyspark.sql.types import StructType, StructField, StringType, LongType
from pyspark.sql.functions import current_timestamp, col, max as spark_max

# Widgets
dbutils.widgets.text("socrata_domain", "opendata.maryland.gov", "Socrata Domain")
dbutils.widgets.text("catalog_name", "open_data_portal", "Catalog Name")
dbutils.widgets.text("schema_name", "meta", "Control Schema Name")
dbutils.widgets.text("table_name", "_socrata_ingestion_queue", "Control Table Name")
dbutils.widgets.text("limit", "", "Dataset Limit (Leave blank for ALL)")
dbutils.widgets.dropdown("force_refresh", "false", ["true", "false"], "Force Full Re-queue (Ignore Historical Watermarks)")
dbutils.widgets.dropdown("verbose", "true", ["true", "false"], "Verbose Logging")

SOCRATA_DOMAIN = dbutils.widgets.get("socrata_domain")
CATALOG_NAME = dbutils.widgets.get("catalog_name")
SCHEMA_NAME = dbutils.widgets.get("schema_name")
TABLE_NAME = dbutils.widgets.get("table_name")
FORCE_REFRESH = dbutils.widgets.get("force_refresh").lower() == "true"
VERBOSE = dbutils.widgets.get("verbose").lower() == "true"

limit_str = dbutils.widgets.get("limit").strip()
DATASET_LIMIT = int(limit_str) if limit_str and limit_str not in ["0", "-1"] else None

QUEUE_TABLE = f"`{CATALOG_NAME}`.`{SCHEMA_NAME}`.`{TABLE_NAME}`"

# --- Hardcoded Exclusion Lists ---
EXCLUDED_DATASET_IDS = {
    # "xxxx-xxxx",  # Specific 4x4 Socrata IDs
}

EXCLUDED_TITLE_KEYWORDS = [
    "md imap",
    "Council on Open Data: Meeting Content",
    "Council on Open Data: Annual Reports"
]

EXCLUDED_ATTRIBUTION_DOMAINS = [
    "mdgeodata.md.gov",
]

def log(*args, **kwargs):
    if VERBOSE:
        print(*args, **kwargs)



In [0]:
# COMMAND ----------
# CELL 2: Read Historical Watermarks & Discover Changed Datasets
SECRET_SCOPE = "socrata_scope"
try:
    APP_TOKEN = dbutils.secrets.get(scope=SECRET_SCOPE, key="app_token")
    log("Successfully retrieved Socrata app token from secret scope.")
except Exception:
    APP_TOKEN = None
    log("Warning: Running unauthenticated. Requests will be subject to throttling.")

client = Socrata(SOCRATA_DOMAIN, APP_TOKEN, timeout=60)

# Retrieve MAX(last_portal_update) per dataset for previous successful runs
existing_watermarks = {}
table_exists = spark.catalog.tableExists(QUEUE_TABLE.replace("`", ""))

if table_exists and not FORCE_REFRESH:
    try:
        prev_df = spark.table(QUEUE_TABLE)
        if "last_portal_update" in prev_df.columns:
            watermark_rows = (prev_df.filter("status = 'SUCCESS' AND last_portal_update IS NOT NULL")
                              .groupBy("dataset_id")
                              .agg(spark_max("last_portal_update").alias("max_portal_update")))
            watermark_rows_collected = watermark_rows.collect()
            existing_watermarks = {
                r["dataset_id"]: str(r["max_portal_update"]).strip() for r in watermark_rows_collected
            }
            log(f"Loaded {len(existing_watermarks)} historical watermarks for incremental comparison.")
    except Exception as e:
        log(f"Warning: Could not read prior watermarks: {e}. Defaulting to full discovery.")

def discover_socrata_catalog(domain: str, max_limit: int = None) -> list:
    log(f"Querying Socrata discovery catalog for {domain} via sodapy...")
    try:
        catalog_results = client.datasets(limit=0)
    except Exception as err:
        raise RuntimeError(f"Failed to query Socrata Discovery API: {err}")

    log(f"Total raw items returned from catalog: {len(catalog_results)}")

    discovered = []
    skipped_unchanged = 0
    excluded_count = 0
    skipped_mdgeodata = 0
    skipped_non_public = 0
    skipped_non_tabular = 0

    for item in catalog_results:
        resource = item.get("resource", {})
        classification = item.get("classification", {})
        dataset_id = resource.get("id")
        title = resource.get("name", dataset_id)

        if not dataset_id:
            continue

        # Exclusion Rule 1: Exact ID
        if dataset_id in EXCLUDED_DATASET_IDS:
            excluded_count += 1
            continue

        # Exclusion Rule 2: Title keywords
        title_lower = title.lower()
        if any(keyword.lower() in title_lower for keyword in EXCLUDED_TITLE_KEYWORDS):
            excluded_count += 1
            continue

        # Exclusion Rule 3: Source attribution domain (mdgeodata)
        attr_link = str(resource.get("attribution_link") or "").lower()
        attr_text = str(resource.get("attribution") or "").lower()
        if any(dom in attr_link or dom in attr_text for dom in EXCLUDED_ATTRIBUTION_DOMAINS):
            skipped_mdgeodata += 1
            continue

        # Inclusion Rule 1: Official Tabular Datasets
        resource_type = resource.get("type", "").lower()
        asset_type = item.get("type", "").lower()
        is_tabular = resource_type in ["dataset"] or asset_type in ["dataset"]
        provenance = resource.get("provenance", "").lower() == "official"

        if not (is_tabular and provenance):
            skipped_non_tabular += 1
            continue

        # Inclusion Rule 2: Public Dataset Validation
        approvals = resource.get("approvals") or item.get("approvals") or [{}]
        first_approval = approvals[0] if isinstance(approvals, list) and approvals else {}
        
        target_audience = (
            first_approval.get("targetAudience") or 
            resource.get("targetAudience") or 
            item.get("targetAudience") or 
            ""
        ).lower()
        
        publication_stage = (
            resource.get("publicationStage") or 
            item.get("publicationStage") or 
            ""
        ).lower()

        # Enforce public and published flags
        if publication_stage and publication_stage != "published":
            skipped_non_public += 1
            continue
        if target_audience and target_audience != "public":
            skipped_non_public += 1
            continue

        # Watermark Check: Skip if portal's update timestamp is not after our last ingest
        portal_updated_at = resource.get("data_updated_at") or resource.get("updatedAt")
        if not FORCE_REFRESH and dataset_id in existing_watermarks:
            prev_portal_update = existing_watermarks[dataset_id]
            if portal_updated_at and portal_updated_at <= prev_portal_update:
                skipped_unchanged += 1
                continue

        category = classification.get("categories", ["Uncategorized"])
        category_str = category[0] if isinstance(category, list) and category else "Uncategorized"

        discovered.append({
            "dataset_id": dataset_id,
            "title": title,
            "category": category_str,
            "last_portal_update": portal_updated_at,
            "status": "QUEUED",
            "records_ingested": None,
            "error_message": None
        })

        if max_limit and len(discovered) >= max_limit:
            log(f"Reached limit cutoff: {max_limit}")
            break

    log(f"Filtered out {skipped_non_tabular} non-tabular or unofficial items.")
    log(f"Filtered out {skipped_non_public} non-public or unpublished datasets.")
    log(f"Filtered out {excluded_count} datasets by ID/title/tag exclusions.")
    log(f"Filtered out {skipped_mdgeodata} datasets matching excluded attribution (mdgeodata).")
    log(f"Skipped {skipped_unchanged} datasets not updated since last successful ingest.")
    return discovered

datasets = discover_socrata_catalog(SOCRATA_DOMAIN, max_limit=DATASET_LIMIT)
log(f"Total datasets newly queued for ingestion: {len(datasets)}")

In [0]:
# CELL 3: Append Discovered Tasks to Historical Log Table
schema = StructType([
    StructField("dataset_id", StringType(), False),
    StructField("title", StringType(), True),
    StructField("last_portal_update", StringType(), True),
    StructField("status", StringType(), True),
    StructField("records_ingested", LongType(), True),
    StructField("error_message", StringType(), True)
])

if datasets:
    new_entries_df = (spark.createDataFrame(datasets, schema=schema)
                      .withColumn("logged_at", current_timestamp()))

    # Always append to maintain the immutable task audit log
    (new_entries_df.write
        .format("delta")
        .mode("append")
        .option("mergeSchema", "true")
        .saveAsTable(QUEUE_TABLE))

    log(f"Successfully appended {len(datasets)} task records into audit log: {QUEUE_TABLE}")
else:
    log("All datasets are up-to-date with portal. No new ingestion tasks logged.")